# Final Test: v1 vs v2 vs Brain-ID

Compare all three embedders on d1/d2/d3 proxies.

In [ ]:
import os
import sys
import numpy as np
from pathlib import Path

# Setup paths
DATA_ROOT = Path("/workspace/data/ehl")
WORK_DIR = Path("/workspace/out")
SHARED = Path("/shared-docker")
AMINE = SHARED / "amine"
MOHAMED = SHARED / "mohamed"

sys.path.insert(0, str(AMINE))
sys.path.insert(0, str(MOHAMED))

from eval_harness import (
    build_image_index, cached_volume, read_csv, mrr,
    SIMULATORS, GRID, N_VAL, SEED
)

print(f"Data: {DATA_ROOT}")
print(f"Grid: {GRID}, N_val: {N_VAL}, Seed: {SEED}")

In [ ]:
# Load data
print("Building image index...")
index = build_image_index(DATA_ROOT)
pairs = read_csv(DATA_ROOT / "dataset1" / "train_pairs.csv")

# Split train/val
rng = np.random.default_rng(SEED)
rng.shuffle(pairs)
val_pairs = pairs[:N_VAL]
train_pairs = pairs[N_VAL:]

print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")

In [ ]:
# Import embedders
from learned_embedder import build as build_v1
from learned_embedder_v2 import build as build_v2

try:
    from brain_id_embedder import build as build_brain_id
    has_brain_id = True
    print("✓ Brain-ID available")
except Exception as e:
    has_brain_id = False
    print(f"✗ Brain-ID import failed: {e}")
    print("  (will still compare v1 and v2)")

print("✓ v1 available")
print("✓ v2 available")

In [ ]:
# Train all embedders
import time

embedders = {}

print("\n" + "="*60)
print("Training V1 (Original)")
print("="*60)
t0 = time.time()
embedders["v1"] = build_v1(train_pairs, index, GRID, cached_volume)
print(f"V1 done in {time.time()-t0:.0f}s\n")

print("="*60)
print("Training V2 (Improved Augmentation)")
print("="*60)
t0 = time.time()
embedders["v2"] = build_v2(train_pairs, index, GRID, cached_volume)
print(f"V2 done in {time.time()-t0:.0f}s\n")

if has_brain_id:
    print("="*60)
    print("Training Brain-ID (Pretrained + Projection Head)")
    print("="*60)
    t0 = time.time()
    try:
        embedders["brain_id"] = build_brain_id(train_pairs, index, GRID, cached_volume)
        print(f"Brain-ID done in {time.time()-t0:.0f}s\n")
    except Exception as e:
        print(f"Brain-ID failed: {e}")
        print("Continuing with v1 and v2 only...\n")

In [ ]:
# Evaluate all on d1/d2/d3
print("\n" + "="*70)
print("EVALUATION RESULTS")
print("="*70)

results = {}

for name, embed_fn in embedders.items():
    print(f"\n{name.upper()}:")
    results[name] = {}
    
    # Load val volumes
    q_vols = [cached_volume(p["query_id"], index[p["query_id"]], GRID) for p in val_pairs]
    g_vols = [cached_volume(p["target_id"], index[p["target_id"]], GRID) for p in val_pairs]
    
    # Evaluate each level
    for level_name, simulator in SIMULATORS.items():
        t0 = time.time()
        lvl_rng = np.random.default_rng(SEED + hash(level_name) % 1000)
        q_sim = [simulator(v, lvl_rng) for v in q_vols]
        g_sim = [simulator(v, lvl_rng) for v in g_vols]
        
        q_emb = np.stack([embed_fn(v) for v in q_sim])
        g_emb = np.stack([embed_fn(v) for v in g_sim])
        
        score = mrr(q_emb, g_emb, np.arange(len(val_pairs)))
        results[name][level_name] = score
        print(f"  {level_name}: {score:.4f} ({time.time()-t0:.1f}s)")
    
    macro = np.mean(list(results[name].values()))
    results[name]["macro"] = macro
    print(f"  MACRO: {macro:.4f}")

In [ ]:
# Summary table
import pandas as pd

df = pd.DataFrame(results).T
df = df[["d1", "d2", "d3", "macro"]]

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(df.to_string())

# Verdict
print("\n" + "="*70)
if "brain_id" in results:
    best = max(results, key=lambda x: results[x]["macro"])
    if best == "brain_id":
        gain = (results["brain_id"]["macro"] - results["v1"]["macro"]) / results["v1"]["macro"] * 100
        print(f"WINNER: Brain-ID with {gain:+.1f}% improvement")
        print(f"V1 macro: {results['v1']['macro']:.4f}")
        print(f"Brain-ID macro: {results['brain_id']['macro']:.4f}")
        print(f"\nReady to submit! Expected Kaggle score: ~0.68-0.72")
    else:
        print(f"V1 still wins. Check Brain-ID implementation.")
else:
    print("Brain-ID not available. Comparing v1 and v2:")
    if results["v2"]["macro"] > results["v1"]["macro"]:
        print(f"V2 wins: {results['v2']['macro']:.4f} vs {results['v1']['macro']:.4f}")
    else:
        print(f"V1 wins: {results['v1']['macro']:.4f} vs {results['v2']['macro']:.4f}")